# 04 — Financial Source

**Objective**: exercise `src/connectors/financial_client.py`, `src/services/financial_metrics.py`, and `src/services/reconciliation.py` end to end — deterministic financial calculations, all seven required risk/quality detectors, and confidence scoring — against both the hand-designed `MockFinancialDataSource` (clean GREEN/AMBER/RED/overrun/missing scenarios) and the real `CSVFinancialDataSource` (data/sample/financial_mock_data.csv).

**Dependencies**: `src/connectors/financial_client.py`, `src/services/financial_metrics.py`, `src/services/reconciliation.py`, `config/financial_field_mapping.yaml`, `config/risk_rules.yaml`.

**Configuration**: no arguments needed — `CSVFinancialDataSource()` and `MockFinancialDataSource()` both have sensible defaults.

In [1]:
import os
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)

from datetime import date
import pandas as pd

from src.connectors.financial_client import CSVFinancialDataSource, MockFinancialDataSource
from src.services import financial_metrics, reconciliation

csv_source = CSVFinancialDataSource()
mock_source = MockFinancialDataSource()
print("Both sources ready.")

Both sources ready.


## Deterministic calculations (Section 5)

`remaining_budget`, `budget_consumption_pct`, and `forecast_variance` are computed once, in Python, by `FinancialRecord.with_calculated_fields()` — never by an LLM. Connector methods return the raw-but-typed record; calculation is a separate, explicit step so the audit trail can show "before" and "after".

In [2]:
raw = csv_source.get_project_finances("10001", "2026-01")
print("RAW (pre-calculation):")
print(f"  approved_budget={raw.approved_budget}  actual_spend={raw.actual_spend}  committed_spend={raw.committed_spend}")
print(f"  remaining_budget={raw.remaining_budget}  budget_consumption_pct={raw.budget_consumption_pct}  (both None — not yet calculated)\n")

calc = raw.with_calculated_fields()
print("CALCULATED:")
print(f"  remaining_budget       = approved - actual - committed = {calc.remaining_budget:,.2f}")
print(f"  budget_consumption_pct = actual / approved * 100        = {calc.budget_consumption_pct:.2f}%")
print(f"  forecast_variance      = approved - forecast             = {calc.forecast_variance:,.2f}")

RAW (pre-calculation):
  approved_budget=60076.96  actual_spend=66923.39  committed_spend=70355.64
  remaining_budget=None  budget_consumption_pct=None  (both None — not yet calculated)

CALCULATED:
  remaining_budget       = approved - actual - committed = -77,202.07
  budget_consumption_pct = actual / approved * 100        = 111.40%
  forecast_variance      = approved - forecast             = -8,476.84


## Sample data demonstrating GREEN / AMBER / RED / overrun / missing

`MockFinancialDataSource` ships with five hand-designed projects, each engineered to land cleanly in one category — the real CSV's numbers are useful precisely because they're messy (see the reconciliation section below), which makes them a poor fixture for demonstrating one clean signal at a time.

In [3]:
rows = []
for project_id in ["MOCK-GREEN", "MOCK-AMBER", "MOCK-RED", "MOCK-OVERRUN", "MOCK-MISSING"]:
    ev = reconciliation.evaluate_financial_record(mock_source, project_id, "2026-06", as_of=date(2026, 6, 20))
    rows.append({
        "project": project_id,
        "consumption_pct": round(ev.record.budget_consumption_pct, 1) if ev.record else None,
        "remaining_budget": round(ev.record.remaining_budget, 2) if ev.record else None,
        "forecast_variance": round(ev.record.forecast_variance, 2) if ev.record else None,
        "financial_risk": ev.financial_risk.severity.value if ev.financial_risk else "N/A",
        "reason_codes": ", ".join(ev.financial_risk.reason_codes) if ev.financial_risk else "",
        "data_quality_flags": ", ".join(r.reason_codes[0] for r in ev.data_quality_risks),
        "confidence": ev.confidence.value,
    })

pd.DataFrame(rows)

,project,consumption_pct,remaining_budget,forecast_variance,financial_risk,reason_codes,data_quality_flags,confidence
0,MOCK-GREEN,45.0,90000.0,5000.0,LOW,,,HIGH CONFIDENCE
1,MOCK-AMBER,80.0,30000.0,2000.0,MEDIUM,HIGH_BUDGET_CONSUMPTION,,HIGH CONFIDENCE
2,MOCK-RED,97.5,-10000.0,-30000.0,HIGH,"FORECAST_OVERRUN, HIGH_BUDGET_CONSUMPTION, NEG...",,HIGH CONFIDENCE
3,MOCK-OVERRUN,53.3,50000.0,-25000.0,HIGH,FORECAST_OVERRUN,,HIGH CONFIDENCE
4,MOCK-MISSING,NaN,NaN,NaN,N/A,,DATA_COMPLETENESS,LOW CONFIDENCE


Note `MOCK-OVERRUN`: consumption is only 53.3% — nowhere near either consumption threshold — yet it's still HIGH, driven entirely by `FORECAST_OVERRUN`. That's the point of checking forecast independently of consumption: it catches trouble *before* it shows up in the spend-to-date numbers.

`MOCK-MISSING` was never in the fixture list at all — looking it up returns `None` (see `ev.record`), which is exactly "missing financial record", not a project with bad numbers.

## Real data: financial reconciliation failure

The CSV's own `remaining_budget` column turns out to use a *different* formula than Section 5's — a real, verifiable data-quality finding, not a synthetic example.

In [4]:
r = csv_source.get_project_finances("10001", "2026-01").with_calculated_fields()
print(f"approved_budget                    = {r.approved_budget:,.2f}")
print(f"actual_spend                       = {r.actual_spend:,.2f}")
print(f"committed_spend                    = {r.committed_spend:,.2f}")
print(f"source_reported_remaining_budget   = {r.source_reported_remaining_budget:,.2f}   (the CSV's own 'remaining_budget' column)")
print(f"this pipeline's remaining_budget   = {r.remaining_budget:,.2f}   (approved - actual - committed, per Section 5)")
print()
print(f"difference = {r.remaining_budget - r.source_reported_remaining_budget:,.2f}  (exactly -committed_spend: "
      f"the source's own figure simply never subtracts committed spend)")

finding = reconciliation.detect_reconciliation_failure(r)
print(f"\nFinding: {finding.description}")
for line in finding.evidence:
    print("  -", line)

approved_budget                    = 60,076.96
actual_spend                       = 66,923.39
committed_spend                    = 70,355.64
source_reported_remaining_budget   = -6,846.43   (the CSV's own 'remaining_budget' column)
this pipeline's remaining_budget   = -77,202.07   (approved - actual - committed, per Section 5)

difference = -70,355.64  (exactly -committed_spend: the source's own figure simply never subtracts committed spend)

Finding: Source-reported remaining budget does not match this pipeline's computed figure
  - computed_remaining_budget (approved - actual - committed) = -77,202.07
  - source_reported_remaining_budget = -6,846.43
  - difference = -70,355.64


In [5]:
# Confirm it's not a one-off — every row in the real dataset has this property.
mismatches = 0
total = 0
for project_id in ["10001", "10002", "10003", "10004", "10005"]:
    for period in [f"2026-{m:02d}" for m in range(1, 9)]:
        rec = csv_source.get_project_finances(project_id, period)
        if rec is None:
            continue
        total += 1
        calc = rec.with_calculated_fields()
        if reconciliation.detect_reconciliation_failure(calc) is not None:
            mismatches += 1
print(f"{mismatches}/{total} rows show a reconciliation mismatch against the source's own remaining_budget figure")

40/40 rows show a reconciliation mismatch against the source's own remaining_budget figure


## Missing financial record: QSR (Jira project 10006)

This is the same unmapped project from Phase 1/the Jira layer — it exists in Jira but was never given a `finance_project_id`. From the financial side, that shows up identically: no record, ever, for any period.

In [6]:
ev = reconciliation.evaluate_financial_record(csv_source, "10006", "2026-08", as_of=date(2026, 9, 15))
print(f"record: {ev.record}")
print(f"financial_risk: {ev.financial_risk}")
print(f"confidence: {ev.confidence}")
for r in ev.data_quality_risks:
    print(f"  [{r.severity.value}] {r.description}  reason_codes={r.reason_codes}")

record: None
financial_risk: None
confidence: ConfidenceLevel.LOW
  [HIGH] No financial record found for 2026-08  reason_codes=['DATA_COMPLETENESS']


## Stale financial reporting period

`detect_stale_reporting_period` compares the SOURCE's latest available period against `as_of` — a threshold of 1080 hours (45 days) accounts for a monthly close's normal lag without flagging every project as stale every single day of the month.

In [7]:
print(f"Latest available period for project 10001: {csv_source.get_latest_reporting_period('10001')}\n")

for label, as_of in [("2 weeks after Aug close", date(2026, 9, 15)), ("7 months after Aug close", date(2027, 3, 1))]:
    ev = reconciliation.evaluate_financial_record(csv_source, "10001", "2026-08", as_of=as_of)
    stale_findings = [r for r in ev.data_quality_risks if "DATA_STALE" in r.reason_codes]
    status = stale_findings[0].description if stale_findings else "not stale"
    print(f"as_of={as_of} ({label}): confidence={ev.confidence.value:16s} -> {status}")

Latest available period for project 10001: 2026-08

as_of=2026-09-15 (2 weeks after Aug close): confidence=MEDIUM CONFIDENCE -> not stale
as_of=2027-03-01 (7 months after Aug close): confidence=LOW CONFIDENCE   -> Latest available financial data (2026-08) is 182 days old


## Spend ahead of delivery — requires delivery data from the Jira layer

This financial layer never fetches delivery progress itself; `delivery_progress_pct` is an optional parameter the caller supplies once the Jira and financial layers are combined (Phase 6). Without it, the check is skipped — not silently treated as "no risk".

In [8]:
r = mock_source.get_project_finances("MOCK-RED", "2026-06").with_calculated_fields()

without_delivery = financial_metrics.detect_spend_ahead_of_delivery(r, None, financial_metrics.load_financial_risk_rules())
print(f"No delivery data supplied -> {without_delivery}  (skipped, not 'no risk')")

with_delivery = financial_metrics.detect_spend_ahead_of_delivery(r, delivery_progress_pct=25.0, rules=financial_metrics.load_financial_risk_rules())
print(f"delivery_progress_pct=25.0 -> {with_delivery.description}")
for line in with_delivery.evidence:
    print("  -", line)

No delivery data supplied -> None  (skipped, not 'no risk')
delivery_progress_pct=25.0 -> Spend-to-progress ratio (3.90x) exceeds the 1.3x threshold
  - budget_consumption_pct=97.5%
  - delivery_progress_pct=25.0%
  - spend_to_progress_ratio=3.90x


## Confidence: completeness + freshness

In [9]:
scenarios = [
    ("fresh + reconciled", reconciliation.compute_confidence(record_found=True, is_stale=False, has_reconciliation_failure=False)),
    ("stale only", reconciliation.compute_confidence(record_found=True, is_stale=True, has_reconciliation_failure=False)),
    ("reconciliation failure only", reconciliation.compute_confidence(record_found=True, is_stale=False, has_reconciliation_failure=True)),
    ("stale AND reconciliation failure", reconciliation.compute_confidence(record_found=True, is_stale=True, has_reconciliation_failure=True)),
    ("record missing entirely", reconciliation.compute_confidence(record_found=False, is_stale=False, has_reconciliation_failure=False)),
]
for label, confidence in scenarios:
    print(f"  {label:38s} -> {confidence.value}")

  fresh + reconciled                     -> HIGH CONFIDENCE
  stale only                             -> MEDIUM CONFIDENCE
  reconciliation failure only            -> MEDIUM CONFIDENCE
  stale AND reconciliation failure       -> LOW CONFIDENCE
  record missing entirely                -> LOW CONFIDENCE


## Portfolio view: real data, full evaluation

In [10]:
rows = []
for project_id in ["10001", "10002", "10003", "10004", "10005", "10006"]:
    latest = csv_source.get_latest_reporting_period(project_id) or "2026-08"
    ev = reconciliation.evaluate_financial_record(csv_source, project_id, latest, as_of=date(2026, 9, 15))
    rows.append({
        "project_id": project_id,
        "period": latest,
        "consumption_pct": round(ev.record.budget_consumption_pct, 1) if ev.record else None,
        "remaining_budget": round(ev.record.remaining_budget, 2) if ev.record else None,
        "financial_risk": ev.financial_risk.severity.value if ev.financial_risk else "N/A",
        "data_quality_flags": len(ev.data_quality_risks),
        "confidence": ev.confidence.value,
    })
pd.DataFrame(rows)

,project_id,period,consumption_pct,remaining_budget,financial_risk,data_quality_flags,confidence
0,10001,2026-08,111.9,-81700.31,HIGH,1,MEDIUM CONFIDENCE
1,10002,2026-08,81.0,-51233.41,HIGH,1,MEDIUM CONFIDENCE
2,10003,2026-08,113.7,-88471.95,HIGH,1,MEDIUM CONFIDENCE
3,10004,2026-08,102.4,-85974.30,HIGH,1,MEDIUM CONFIDENCE
4,10005,2026-08,106.2,-46086.94,HIGH,1,MEDIUM CONFIDENCE
5,10006,2026-08,NaN,NaN,N/A,1,LOW CONFIDENCE


## Validation checks

- [x] `remaining_budget` / `budget_consumption_pct` / `forecast_variance` computed only in Python, never returned pre-filled by a connector
- [x] Divide-by-zero on `approved_budget == 0` returns `None`, not an exception or a fabricated 0%
- [x] All 7 required detections demonstrated: forecast overrun, high consumption, negative remaining, spend ahead of delivery, missing record, stale period, reconciliation failure
- [x] GREEN / AMBER / RED / forecast-overrun / missing all reproduced cleanly via `MockFinancialDataSource`
- [x] Reconciliation failure proven on 100% of the real CSV's rows, with the exact discrepancy (`-committed_spend`) explained
- [x] Confidence downgrades correctly for missing / stale / inconsistent data, independently and in combination
- [x] Every result carries `retrieved_timestamp` and `reporting_period`

## Testing

`tests/test_financial_metrics.py` (25 tests) and `tests/test_reconciliation.py` (31 tests) — `pytest tests/test_financial_metrics.py tests/test_reconciliation.py -v`.

## Next step

Phase 5/6 territory: cross-source project unification (joining this financial view to the Jira view via `config/project_mapping.yaml`) and the combined delivery+financial risk matrix. Stopping here per the current task scope — financial intelligence layer validated.